# A3-05 · Combined final report (development and holdout, kept apart)

Requires a verifying freeze and every frozen holdout unit completed. Writes a development report (all of Exp 0-8a,
comparisons, selection checks, interpretation) and a separate descriptive holdout report to new folders of `A3_final/reports/`.
The holdout report shows holdout metrics only, plus the development AUCs of the frozen records in a labelled comparison
table; it contains no test and selects nothing.

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook again
and *Run all* again: finished work is kept on Drive. Everything A3 writes goes to `<outputs>/FINAL_v1/amendments/A3_final/`
(plus, once, the two frozen records created by the freeze); no completed result, pin, QC decision or store is changed.

**Holdout policy.** The 13-participant holdout was inspected during earlier experiments (LX / WX suites), so it is no longer an untouched test set. In this release it is evaluated once per final configuration and reported descriptively; it is never used to select architectures, graph constructions, k, atlas resolution, QC thresholds or any other setting (selection reads development CV results only).

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the A3 code, installs the requirements. =====
UPDATE_CODE = False   # leave False. True only to move the A3 code pin (<outputs>/FINAL_v1/amendments/A3_final/CODE_COMMIT.txt)
BRANCH = 'claude/a3-final-analysis'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])   # full history: the pins are ancestors
os.chdir('/content/repo')
_run([sys.executable, 'scripts/a3_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')

from adgraph.colab import sh as _sh

def sh(cmd, cwd='/content/repo'):  # every script runs with the file backend
    os.environ['MPLBACKEND'] = 'Agg'
    return _sh(cmd, cwd)

from adgraph.config import load_paths
PATHS = load_paths('/content/repo/configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
A3 = FINAL + '/amendments/A3_final'
print('A3 outputs on Drive:', A3)

## 1. Freeze verification and holdout completeness

In [ ]:
sh("python scripts/a3.py verify")
sh("python scripts/a3.py holdout-status")
from adgraph.a3 import holdout
st = holdout.status(OUT)
assert (st['holdout_state'] == 'completed').all(), st[st['holdout_state'] != 'completed']

## 2. Development report and holdout report

In [ ]:
sh("python scripts/a3.py report")
sh("python scripts/a3.py holdout-report")

In [ ]:
import glob, json
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_rows', 400, 'display.width', 250, 'display.max_columns', 40)

def latest(pattern):
    found = sorted(glob.glob(pattern))
    return found[-1] if found else None
def show_latest(kind):
    d = latest(A3 + f'/reports/{kind}_*')
    print(d)
    return d
d = show_latest('development')
print(open(d + '/interpretation.md').read())
h = show_latest('holdout')
print(open(h + '/holdout_results.md').read())
for png in sorted(glob.glob(d + '/figures/*.png')) + sorted(glob.glob(h + '/*.png')):
    display(Image(filename=png))